# Rol 2 — Síntesis del instrumento y función de fitness

Este notebook utiliza la representación del cromosoma y la melodía
extraída por el Rol 1 para generar un sonido de flauta sintetizada.

El instrumento se controla mediante ocho genes: amplitudes de tres
armónicos, ataque, decaimiento, sostenido, relajación y ruido de soplo.

Se comprobarán la síntesis, los silencios y las funciones de evaluación
mediante STFT en decibelios y croma.

Las primeras comparaciones utilizan audios sintéticos para verificar
el funcionamiento del código. La evaluación del parecido con una
flauta requiere una referencia instrumental independiente.


## 1. Configuración del entorno

Se localiza la raíz del proyecto y se importan las funciones compartidas.

Del Rol 1 se reutilizan la lectura del CSV, la representación del
cromosoma y la conversión a parámetros físicos. Del Rol 2 se importan
las funciones de síntesis, exportación y evaluación.

In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Audio, display

RAIZ = Path.cwd().resolve()

if not (RAIZ / "src" / "representacion.py").is_file():
    RAIZ = RAIZ.parent

if not (RAIZ / "src" / "representacion.py").is_file():
    raise FileNotFoundError(
        "Ejecuta el notebook desde la raíz del proyecto "
        "o desde la carpeta notebooks/."
    )

if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import src.representacion as rep

from src.extraccion import (
    cargar_melodia_csv,
    nombre_nota_midi,
)

from src.sintesis import (
    FRECUENCIA_MUESTREO,
    sintetizar_instrumento,
    guardar_wav,
)

from src.fitness import (
    evaluar_audio,
    evaluar_individuo,
)

print(f"Raíz del proyecto: {RAIZ}")
print(f"Frecuencia de muestreo: {FRECUENCIA_MUESTREO} Hz")
print(f"Genes definidos por el Rol 1: {rep.num_genes()}")
# Carpeta existente para las salidas.
CARPETA_AUDIO = RAIZ / "resultados" / "audios"

Raíz del proyecto: C:\Users\Asus\Downloads\Sistemas inteligentes\silbido-a-instrumento-prototipo-AG-IA
Frecuencia de muestreo: 22050 Hz
Genes definidos por el Rol 1: 8


## 2. Carga de la melodía extraída por el Rol 1

Se utiliza el archivo datos/procesados/melodia_01.csv, que contiene
una nota MIDI y una duración en segundos por evento.

El valor -1 representa un silencio. La melodía permanece fija:
el algoritmo genético modificará los parámetros del instrumento,
no las notas ni sus duraciones.

In [3]:
ruta_csv = RAIZ / "datos" / "procesados" / "melodia_01.csv"

notas, duraciones = cargar_melodia_csv(ruta_csv)

if notas.size == 0 or notas.size != duraciones.size:
    raise ValueError(
        "La melodía está vacía o no contiene una duración por evento."
    )

if not np.all(np.isfinite(duraciones)) or np.any(duraciones <= 0):
    raise ValueError("Las duraciones deben ser positivas y finitas.")

inicio_eventos = np.concatenate((
    [0.0],
    np.cumsum(duraciones)[:-1],
))

tabla_melodia = pd.DataFrame({
    "nota_midi": notas,
    "nombre": [nombre_nota_midi(int(n)) for n in notas],
    "inicio_s": inicio_eventos,
    "duracion_s": duraciones,
})

display(tabla_melodia)

print(f"Eventos: {len(notas)}")
print(f"Notas: {np.count_nonzero(notas != -1)}")
print(f"Silencios: {np.count_nonzero(notas == -1)}")
print(f"Duración total del CSV: {duraciones.sum():.3f} s")

,nota_midi,nombre,inicio_s,duracion_s
0,-1,Silencio (-1),0.000,0.128
1,93,La6 (93),0.128,0.244
2,96,Do7 (96),0.372,0.104
3,-1,Silencio (-1),0.476,0.139
4,92,Sol#6 (92),0.615,0.244
5,95,Si6 (95),0.859,0.255
6,92,Sol#6 (92),1.114,0.279
7,-1,Silencio (-1),1.393,0.035
8,95,Si6 (95),1.428,0.232
9,-1,Silencio (-1),1.660,0.035


Eventos: 16
Notas: 11
Silencios: 5
Duración total del CSV: 2.705 s


## 3. Cromosoma y parámetros del instrumento

Se obtiene el cromosoma inicial desde los valores por defecto del
Rol 1. Sus ocho genes están normalizados entre 0 y 1.

La función decodificar() transforma esos valores a amplitudes y
tiempos físicos. Los valores por defecto constituyen una configuración
inicial de prueba; no son parámetros optimizados ni una referencia real.

In [4]:
GENES_ESPERADOS = (
    "amp_arm1",
    "amp_arm2",
    "amp_arm3",
    "ataque_s",
    "decaimiento_s",
    "sostenido_rel",
    "relajacion_s",
    "ruido_soplo",
)

nombres_actuales = tuple(rep.obtener_nombres_genes())

if nombres_actuales != GENES_ESPERADOS:
    raise ValueError(
        "El esquema del Rol 1 cambió. "
        "Revisa la compatibilidad de síntesis antes de continuar.\n"
        f"Esperado: {GENES_ESPERADOS}\n"
        f"Recibido: {nombres_actuales}"
    )

gen_prueba = rep.obtener_valores_default(normalizado=True)

valido, mensaje = rep.validar_individuo(gen_prueba)

if not valido:
    raise ValueError(mensaje)

parametros = rep.decodificar(gen_prueba)

tabla_genes = pd.DataFrame([
    {
        "indice": indice,
        "gen": especificacion.nombre,
        "valor_normalizado": gen_prueba[indice],
        "valor_fisico": parametros[especificacion.nombre],
        "unidad": especificacion.unidad,
    }
    for indice, especificacion in enumerate(rep.GENES_SCHEMA)
])

display(tabla_genes)

,indice,gen,valor_normalizado,valor_fisico,unidad
0,0,amp_arm1,0.888889,0.90,amplitud rel.
1,1,amp_arm2,0.312500,0.25,amplitud rel.
2,2,amp_arm3,0.160000,0.08,amplitud rel.
3,3,ataque_s,0.183673,0.05,segundos
4,4,decaimiento_s,0.254237,0.08,segundos
5,5,sostenido_rel,0.777778,0.80,amplitud rel.
6,6,relajacion_s,0.186441,0.06,segundos
7,7,ruido_soplo,0.160000,0.04,amplitud rel.


## 4. Síntesis de la melodía

Cada nota se genera sumando tres armónicos y ruido de soplo.
La envolvente ADSR controla la evolución de su amplitud.

Todas las etapas de la envolvente quedan dentro de la duración
del evento. Si no caben, sus tiempos se reducen proporcionalmente.

Se emplea una semilla fija para el ruido y una ganancia fija para
todos los individuos. Los silencios generan muestras de valor cero.

In [5]:
import importlib
import inspect
import src.sintesis as modulo_sintesis
import src.fitness as modulo_fitness

# Recargar el código guardado en disco.
importlib.invalidate_caches()
importlib.reload(modulo_sintesis)
importlib.reload(modulo_fitness)

# Actualizar también las funciones importadas en el notebook.
sintetizar_instrumento = modulo_sintesis.sintetizar_instrumento
guardar_wav = modulo_sintesis.guardar_wav
FRECUENCIA_MUESTREO = modulo_sintesis.FRECUENCIA_MUESTREO

evaluar_audio = modulo_fitness.evaluar_audio
evaluar_individuo = modulo_fitness.evaluar_individuo

print("Archivo cargado:", modulo_sintesis.__file__)
print("Firma:", inspect.signature(sintetizar_instrumento))
print("Genes recibidos:", len(gen_prueba))

assert "semilla" in inspect.signature(
    sintetizar_instrumento
).parameters, (
    "El archivo guardado sigue siendo la versión antigua. "
    "Reemplaza el contenido del archivo indicado arriba "
    "por la versión de ocho genes y guárdalo."
)

assert len(gen_prueba) == 8

print("Versión de ocho genes cargada correctamente.")

Archivo cargado: C:\Users\Asus\Downloads\Sistemas inteligentes\silbido-a-instrumento-prototipo-AG-IA\src\sintesis.py
Firma: (notas, duraciones, gen, *, semilla=2026)
Genes recibidos: 8
Versión de ocho genes cargada correctamente.


In [6]:
audio = sintetizar_instrumento(
    notas,
    duraciones,
    gen_prueba,
)

muestras_esperadas = int(round(
    duraciones.sum() * FRECUENCIA_MUESTREO
))

assert audio.ndim == 1
assert len(audio) == muestras_esperadas
assert np.all(np.isfinite(audio))
assert np.max(np.abs(audio)) <= 1.0

print(f"Muestras generadas: {len(audio)}")
print(f"Duración: {len(audio) / FRECUENCIA_MUESTREO:.6f} s")
print(f"Amplitud máxima: {np.max(np.abs(audio)):.4f}")

display(
    Audio(
        audio,
        rate=FRECUENCIA_MUESTREO,
        normalize=False,
    )
)

Muestras generadas: 59645
Duración: 2.704989 s
Amplitud máxima: 0.3626


## 5. Comprobación de reproducibilidad y silencios

Se verifica que la misma melodía y el mismo cromosoma produzcan
exactamente el mismo audio al utilizar la semilla predeterminada.

También se comprueba que los eventos identificados como silencios
contengan únicamente muestras cero.

In [7]:
audio_repetido = sintetizar_instrumento(
    notas,
    duraciones,
    gen_prueba,
)

assert np.array_equal(audio, audio_repetido), (
    "La síntesis no está produciendo resultados reproducibles."
)

fronteras = np.rint(
    np.concatenate(([0.0], np.cumsum(duraciones)))
    * FRECUENCIA_MUESTREO
).astype(int)

for indice, nota in enumerate(notas):
    if nota == -1:
        fragmento = audio[
            fronteras[indice]:fronteras[indice + 1]
        ]

        assert np.all(fragmento == 0.0), (
            f"El silencio del evento {indice} contiene sonido."
        )

print("Reproducibilidad: correcta.")
print("Silencios: correctos.")

Reproducibilidad: correcta.
Silencios: correctos.


## 6. Exportación del audio inicial

Se guarda la melodía sintetizada en formato WAV mono, a 22050 Hz
y PCM de 16 bits.

Este archivo corresponde al cromosoma por defecto. Todavía no es
el mejor individuo evolucionado ni una salida de la generación cero
de una población.

In [8]:
ruta_audio = guardar_wav(
    CARPETA_AUDIO / "sintesis_melodia_rol1.wav",
    audio,
)

print(f"Audio guardado: {ruta_audio.relative_to(RAIZ)}")

Audio guardado: resultados\audios\sintesis_melodia_rol1.wav


## 7. Métricas de evaluación

### Métrica principal: STFT en decibelios

Se calcula el error absoluto medio entre las magnitudes de los
espectrogramas expresadas en dB, utilizando una referencia fija
de amplitud 1 y un piso de -80 dB.

El error se transforma en aptitud mediante:

fitness_stft = 1 / (1 + error_stft_db / 10)

El algoritmo genético maximiza esta aptitud. Una coincidencia exacta
obtiene 1. El factor 10 controla la escala de la transformación;
no representa un umbral validado de calidad perceptual.

### Métrica alternativa: croma

Se compara la distribución de energía en las doce clases de altura
mediante similitud coseno. La aptitud está entre 0 y 1.

El croma puede distinguir poco entre instrumentos que interpretan
las mismas notas, por lo que se analizará junto con la métrica STFT.

### Condiciones de comparación

Los audios deben ser mono, tener una frecuencia de muestreo de
22050 Hz, igual longitud y eventos temporalmente alineados.

No se utiliza MSE de la onda cruda porque diferencias pequeñas
de fase pueden producir errores grandes entre sonidos similares.

## 8. Comprobación de las métricas con un audio idéntico

Se compara el audio sintetizado consigo mismo.

El resultado esperado es error STFT igual a cero y ambas aptitudes
iguales a uno. Esta comprobación verifica la implementación;
no mide parecido con una flauta real.

In [9]:
resultado_identico = evaluar_audio(audio, audio)

assert np.isclose(resultado_identico["error_stft_db"], 0.0)
assert np.isclose(resultado_identico["fitness_stft"], 1.0)
assert np.isclose(resultado_identico["fitness_croma"], 1.0)

display(pd.DataFrame([resultado_identico]))

,error_stft_db,fitness_stft,fitness_croma
0,0.0,1.0,1.0


## 9. Comparación técnica al modificar un gen

Se copia el cromosoma inicial y se modifica únicamente la amplitud
del segundo armónico.

Las notas, duraciones y los demás genes permanecen constantes.
Ambos audios se comparan para comprobar que el fitness espectral
detecta el cambio introducido.

In [10]:
gen_distinto = gen_prueba.copy()

indice_armonico_2 = rep.obtener_nombres_genes().index("amp_arm2")
gen_distinto[indice_armonico_2] = 0.95

audio_distinto = sintetizar_instrumento(
    notas,
    duraciones,
    gen_distinto,
)

resultado_distinto = evaluar_audio(
    audio_distinto,
    audio,
)

assert resultado_distinto["error_stft_db"] > 0.0

tabla_comprobacion = pd.DataFrame([
    {
        "comparacion": "Audio consigo mismo",
        **resultado_identico,
    },
    {
        "comparacion": "Segundo armónico modificado",
        **resultado_distinto,
    },
])

display(tabla_comprobacion)

print("Audio con el segundo armónico modificado:")

display(
    Audio(
        audio_distinto,
        rate=FRECUENCIA_MUESTREO,
        normalize=False,
    )
)

,comparacion,error_stft_db,fitness_stft,fitness_croma
0,Audio consigo mismo,0.000000,1.000000,1.000000
1,Segundo armónico modificado,0.149971,0.985224,0.999958


Audio con el segundo armónico modificado:


## 10. Comprobación de la interfaz de evaluación de individuos

La función evaluar_individuo() recibe un cromosoma, la melodía,
una referencia y el nombre de la métrica. Sintetiza el individuo
y devuelve una aptitud numérica que el AG debe maximizar.

En esta comprobación se utiliza el audio inicial como referencia
sintética temporal. La evaluación instrumental deberá utilizar
posteriormente una grabación independiente de flauta.

In [11]:
aptitud = evaluar_individuo(
    gen=gen_distinto,
    notas=notas,
    duraciones=duraciones,
    referencia=audio,
    metrica="stft",
)

assert np.isclose(
    aptitud,
    resultado_distinto["fitness_stft"],
)

print(f"Aptitud devuelta al AG: {aptitud:.6f}")

Aptitud devuelta al AG: 0.985224


## 11. Estado de la integración y próximos experimentos

Las celdas anteriores conectan la representación de ocho genes y
la melodía del Rol 1 con la síntesis y las métricas del Rol 2.

Si las verificaciones pasan, se comprueban:

- La generación de audio con los ocho genes.
- La conservación de la duración definida por el CSV.
- Los silencios absolutos.
- La reproducibilidad de la síntesis.
- La evaluación de audios idénticos y de un cromosoma modificado.

Estas verificaciones no demuestran todavía semejanza instrumental.

## 12. Referencia instrumental de flauta

Se prepara una referencia independiente utilizando grabaciones
reales de flauta sin vibrato de University of Iowa Musical
Instrument Samples.

El montaje utiliza las notas, duraciones y silencios del CSV
entregado por el Rol 1. No se sustituye la melodía detectada
por la secuencia orientativa del diseño inicial.

Las muestras originales se conservan en
datos/audio/referencias/originales/.

La referencia de trabajo y sus metadatos se guardan en
datos/audio/referencias/.

Los comienzos de las notas se estiman mediante energía y deben
revisarse mediante escucha. Se aplican transiciones de 5 ms
para reducir discontinuidades en los bordes.

Las fuentes y el procesamiento quedan registrados en un JSON.
Las condiciones de uso deben revisarse antes de distribuir
las muestras o los audios derivados.

In [12]:
import json
import re

from html import unescape
from math import gcd
from urllib.parse import urljoin, urlsplit, unquote, quote
from urllib.request import Request, urlopen

import soundfile as sf
from scipy.signal import resample_poly


# Carpeta de resultados ya existente.
CARPETA_AUDIO = RAIZ / "resultados" / "audios"

if not CARPETA_AUDIO.is_dir():
    raise FileNotFoundError(
        f"No existe la carpeta de resultados: {CARPETA_AUDIO}"
    )


PAGINA_FLAUTA = (
    "https://theremin.music.uiowa.edu/"
    "MIS-Pitches-2012/MISFlute2012.html"
)

ORIGINALES = (
    RAIZ / "datos" / "audio" / "referencias" / "originales"
)

# Esta carpeta corresponde a las entradas originales,
# no a una nueva carpeta de resultados.
ORIGINALES.mkdir(parents=True, exist_ok=True)

CLASES_NOTAS = [
    "C", "Db", "D", "Eb", "E", "F",
    "Gb", "G", "Ab", "A", "Bb", "B",
]


def descargar_bytes(url):
    """Descarga una página o muestra instrumental."""
    url_codificada = quote(url, safe=":/%?=&")

    peticion = Request(
        url_codificada,
        headers={"User-Agent": "Mozilla/5.0"},
    )

    with urlopen(peticion, timeout=60) as respuesta:
        return respuesta.read()

# Reutilizar el catálogo local cuando cubre las notas del CSV.
# Las muestras existentes se leen sin descargarlas nuevamente.

ruta_catalogo = (
    ORIGINALES.parent / "flauta_referencia_01.json"
)

notas_necesarias_catalogo = {
    int(nota)
    for nota in notas
    if nota != -1
}

catalogo = {}

if ruta_catalogo.is_file():
    catalogo = json.loads(
        ruta_catalogo.read_text(encoding="utf-8")
    )

fuentes_cache = catalogo.get("muestras", [])

notas_disponibles_catalogo = {
    int(fuente["midi"])
    for fuente in fuentes_cache
}

catalogo_completo = (
    notas_necesarias_catalogo
    .issubset(notas_disponibles_catalogo)
)

if catalogo_completo:
    enlaces = [
        fuente["url"]
        for fuente in fuentes_cache
    ]

    print("Catálogo local de fuentes reutilizado.")

else:
    html_flauta = descargar_bytes(
        PAGINA_FLAUTA
    ).decode(
        "utf-8",
        errors="replace",
    )

    enlaces = [
        urljoin(
            PAGINA_FLAUTA,
            unescape(href),
        )
        for href in re.findall(
            r"""href\s*=\s*["']([^"']+)["']""",
            html_flauta,
            flags=re.IGNORECASE,
        )
    ]

    print("Catálogo consultado en la página original.")

Catálogo local de fuentes reutilizado.


## 13. Comparación de STFT y croma

Se evalúan el cromosoma por defecto y cinco cromosomas
aleatorios válidos generados mediante la representación
del Rol 1.

La generación de candidatos utiliza la semilla 42.
La síntesis conserva su semilla predeterminada de ruido,
independiente de la generación de candidatos.

Todos los candidatos interpretan la misma melodía y se
comparan contra la misma referencia instrumental.

Se registran el error espectral, las aptitudes y la posición
de cada candidato en ambas métricas. Las aptitudes se maximizan.

Estos seis candidatos no constituyen una población evolucionada.
El resultado no demuestra mejoras obtenidas mediante un AG.

In [15]:
import soundfile as sf

ruta_referencia = (
    RAIZ
    / "datos"
    / "audio"
    / "referencias"
    / "flauta_referencia_01.wav"
)

if not ruta_referencia.is_file():
    raise FileNotFoundError(
        "No existe la referencia de flauta. "
        "Ejecuta primero la celda completa de la sección 12."
    )

referencia, sr_ref = sf.read(
    ruta_referencia,
    dtype="float32",
)

if sr_ref != FRECUENCIA_MUESTREO:
    raise ValueError(
        f"La referencia está a {sr_ref} Hz; "
        f"se requieren {FRECUENCIA_MUESTREO} Hz."
    )

if referencia.ndim != 1:
    raise ValueError("La referencia debe ser mono.")

if not np.all(np.isfinite(referencia)):
    raise ValueError("La referencia contiene valores no finitos.")

if referencia.size < 2048:
    raise ValueError("La referencia debe tener al menos 2048 muestras.")

if np.max(np.abs(referencia)) <= 1e-8:
    raise ValueError("La referencia es completamente silenciosa.")

audio_comprobacion = sintetizar_instrumento(
    notas,
    duraciones,
    gen_prueba,
)

if len(referencia) != len(audio_comprobacion):
    raise ValueError(
        "La referencia no tiene la longitud de la melodía actual. "
        "Ejecuta la sección 12 para reconstruirla desde el CSV."
    )

print(
    f"Referencia cargada: {len(referencia)} muestras, "
    f"{sr_ref} Hz."
)

SEMILLA_CANDIDATOS = 42

rng_comparacion = np.random.default_rng(
    SEMILLA_CANDIDATOS
)

candidatos = {
    "default": gen_prueba.copy(),
}

for indice in range(5):
    candidatos[f"aleatorio_{indice + 1}"] = (
        rep.generar_individuo_aleatorio(
            rng=rng_comparacion,
        )
    )


filas = []

for nombre, cromosoma in candidatos.items():
    valido, mensaje = rep.validar_individuo(
        cromosoma
    )

    if not valido:
        raise ValueError(
            f"Candidato {nombre} inválido: {mensaje}"
        )

    senal = sintetizar_instrumento(
        notas,
        duraciones,
        cromosoma,
    )

    assert len(senal) == len(referencia)

    resultados = evaluar_audio(
        senal,
        referencia,
    )

    filas.append({
        "candidato": nombre,
        **resultados,
        **{
            f"gen_{nombre_gen}": float(valor)
            for nombre_gen, valor in zip(
                rep.obtener_nombres_genes(),
                cromosoma,
            )
        },
    })


comparacion = pd.DataFrame(filas)

for metrica in ("stft", "croma"):
    comparacion[f"puesto_{metrica}"] = (
        comparacion[f"fitness_{metrica}"].rank(
            ascending=False,
            method="min",
        )
    )

ruta_comparacion = (
    CARPETA_AUDIO / "comparacion_metricas.csv"
)

comparacion.to_csv(
    ruta_comparacion,
    index=False,
)

display(comparacion[[
    "candidato",
    "error_stft_db",
    "fitness_stft",
    "fitness_croma",
    "puesto_stft",
    "puesto_croma",
]])

print(
    "Comparación guardada:",
    ruta_comparacion.relative_to(RAIZ),
)

Referencia cargada: 59645 muestras, 22050 Hz.


,candidato,error_stft_db,fitness_stft,fitness_croma,puesto_stft,puesto_croma
0,default,4.714410,0.679606,0.914508,1.0,2.0
1,aleatorio_1,9.417012,0.515012,0.906468,5.0,3.0
2,aleatorio_2,5.495775,0.645337,0.920207,2.0,1.0
3,aleatorio_3,9.062671,0.524585,0.855770,4.0,6.0
4,aleatorio_4,9.984811,0.500380,0.870044,6.0,5.0
5,aleatorio_5,8.288302,0.546798,0.893893,3.0,4.0


Comparación guardada: resultados\audios\comparacion_metricas.csv


## 14. Comparación de sensibilidad utilizando el módulo del Rol 1

Se reutilizan barrido_sensibilidad_todos() y
calcular_ranking_sensibilidad() desde src/sensibilidad.py.

El análisis utiliza 51 valores normalizados por cada uno de
los ocho genes: 408 evaluaciones. En cada barrido se modifica
un gen y los restantes permanecen en el cromosoma por defecto.

Las notas, duraciones, referencia y semilla de síntesis
permanecen constantes.

Se comparan los rankings de sensibilidad de STFT y croma.
El rango de variación de una métrica describe su respuesta
al gen, pero no demuestra superioridad perceptual frente
a otra métrica.

Los resultados se calculan nuevamente con los datos de esta
ejecución. No se presentan los CSV guardados por el Rol 1
como resultados nuevos.

Se conserva el gráfico individual de amp_arm2 para ilustrar
un corte de la superficie de fitness.

In [16]:
import matplotlib.pyplot as plt

from src.sensibilidad import (
    barrido_sensibilidad_todos,
    calcular_ranking_sensibilidad,
)


PUNTOS_POR_GEN = 51
GEN_BARRIDO = "amp_arm2"

# Reutilizar el cálculo compartido del Rol 1.
sensibilidad = barrido_sensibilidad_todos(
    notas=notas,
    duraciones=duraciones,
    referencia_audio=referencia,
    gen_base=gen_prueba.copy(),
    num_puntos=PUNTOS_POR_GEN,
)

assert len(sensibilidad) == (
    rep.num_genes() * PUNTOS_POR_GEN
)

columnas_numericas = [
    "valor_normalizado",
    "valor_fisico",
    "error_stft_db",
    "fitness_stft",
    "fitness_croma",
]

assert np.all(np.isfinite(
    sensibilidad[columnas_numericas].to_numpy()
))

assert sensibilidad.groupby(
    "gen_nombre"
).size().eq(PUNTOS_POR_GEN).all()

ranking_sensibilidad = calcular_ranking_sensibilidad(
    sensibilidad
)

# El módulo del Rol 1 ya calcula el ranking STFT.
# Añadir el ranking de croma para contrastar ambos.
ranking_sensibilidad["ranking_croma"] = (
    ranking_sensibilidad["delta_fitness_croma"]
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int)
)

sensibilidad.to_csv(
    CARPETA_AUDIO / "rol2_dia3_sensibilidad_metricas.csv",
    index=False,
)

ranking_sensibilidad.to_csv(
    CARPETA_AUDIO / "rol2_dia3_ranking_metricas.csv",
    index=False,
)

display(ranking_sensibilidad[[
    "gen_nombre",
    "ranking_stft",
    "delta_fitness_stft",
    "ranking_croma",
    "delta_fitness_croma",
]])

# Mantener estos nombres para que el registro existente
# del notebook siga siendo compatible.
barrido = (
    sensibilidad.loc[
        sensibilidad["gen_nombre"] == GEN_BARRIDO
    ]
    .sort_values("valor_normalizado")
    .rename(columns={
        "valor_normalizado": "gen_normalizado",
        "valor_fisico": "amplitud_fisica",
    })
    .copy()
)

barrido.to_csv(
    CARPETA_AUDIO / "superficie_amp_arm2.csv",
    index=False,
)

fig, ejes = plt.subplots(
    1,
    2,
    figsize=(11, 4),
)

for eje, metrica in zip(
    ejes,
    ("stft", "croma"),
):
    eje.plot(
        barrido["gen_normalizado"],
        barrido[f"fitness_{metrica}"],
        marker=".",
    )

    eje.set(
        title=f"Fitness {metrica.upper()}",
        xlabel=f"Gen {GEN_BARRIDO} normalizado",
        ylabel="Aptitud — mayor es mejor",
    )

    eje.grid(alpha=0.3)

fig.tight_layout()

fig.savefig(
    CARPETA_AUDIO / "superficie_amp_arm2.png",
    dpi=180,
    bbox_inches="tight",
)

# src.sensibilidad configura Matplotlib con backend Agg.
# Mostrar explícitamente la figura dentro del notebook.
display(fig)
plt.close(fig)

print(
    f"Evaluaciones del barrido: {len(sensibilidad)}"
)

print(
    "Resultados propios del Rol 2 guardados en:",
    CARPETA_AUDIO.relative_to(RAIZ),
)

,gen_nombre,ranking_stft,delta_fitness_stft,ranking_croma,delta_fitness_croma
0,ruido_soplo,1,0.262472,8,0.000379
1,sostenido_rel,2,0.040001,3,0.043348
2,relajacion_s,3,0.018473,1,0.064137
3,decaimiento_s,4,0.015179,6,0.001233
4,ataque_s,5,0.013708,2,0.057390
5,amp_arm3,6,0.009301,4,0.015539
6,amp_arm2,7,0.006564,7,0.000663
7,amp_arm1,8,0.002802,5,0.002066


<Figure size 1100x400 with 2 Axes>

Evaluaciones del barrido: 408
Resultados propios del Rol 2 guardados en: resultados\audios


## 15. Comparación auditiva de los candidatos seleccionados

Se exportan la referencia, el cromosoma por defecto y los
candidatos seleccionados por STFT y croma.

Esto permite escuchar si las diferencias de selección
se corresponden con diferencias perceptibles de timbre
o articulación.

Las reproducciones no normalizan automáticamente el volumen.
La evaluación auditiva debe considerar que los niveles
de amplitud pueden influir en la percepción.

Estos candidatos proceden de la comparación de seis individuos.
Aunque el proyecto ya dispone de ejecutar_ag(), esta sección
no ejecuta una evolución.

Si dos criterios seleccionan el mismo cromosoma, sus audios
serán iguales.

In [17]:
nombre_mejor = str(comparacion.loc[
    comparacion["fitness_stft"].idxmax(),
    "candidato",
])

nombre_mejor_croma = str(comparacion.loc[
    comparacion["fitness_croma"].idxmax(),
    "candidato",
])

gen_mejor = candidatos[nombre_mejor].copy()
gen_mejor_croma = candidatos[nombre_mejor_croma].copy()

audio_default = sintetizar_instrumento(
    notas, duraciones, gen_prueba
)

audio_mejor = sintetizar_instrumento(
    notas, duraciones, gen_mejor
)

audio_mejor_croma = sintetizar_instrumento(
    notas, duraciones, gen_mejor_croma
)

audios_exportados = []

for nombre, senal in {
    "referencia_flauta": referencia,
    "sintesis_default": audio_default,
    "mejor_candidato_evaluado": audio_mejor,
    "mejor_candidato_croma": audio_mejor_croma,
}.items():
    ruta = guardar_wav(
        CARPETA_AUDIO / f"{nombre}.wav",
        senal,
    )

    informacion = sf.info(ruta)

    assert informacion.samplerate == FRECUENCIA_MUESTREO
    assert informacion.channels == 1
    assert informacion.subtype == "PCM_16"
    assert informacion.frames == len(referencia)

    audios_exportados.append({
        "archivo": str(ruta.relative_to(RAIZ)),
        "muestras": informacion.frames,
        "duracion_s": informacion.duration,
    })

    print(nombre)

    display(Audio(
        senal,
        rate=FRECUENCIA_MUESTREO,
        normalize=False,
    ))

np.save(
    CARPETA_AUDIO / "gen_mejor_candidato_evaluado.npy",
    gen_mejor,
)

np.save(
    CARPETA_AUDIO / "gen_mejor_candidato_croma.npy",
    gen_mejor_croma,
)

display(pd.DataFrame(audios_exportados))

print("Seleccionado por STFT:", nombre_mejor)
print("Seleccionado por croma:", nombre_mejor_croma)
print("No se ejecutó el AG en esta comparación.")

referencia_flauta


sintesis_default


mejor_candidato_evaluado


mejor_candidato_croma


,archivo,muestras,duracion_s
0,resultados\audios\referencia_flauta.wav,59645,2.704989
1,resultados\audios\sintesis_default.wav,59645,2.704989
2,resultados\audios\mejor_candidato_evaluado.wav,59645,2.704989
3,resultados\audios\mejor_candidato_croma.wav,59645,2.704989


Seleccionado por STFT: default
Seleccionado por croma: aleatorio_2
No se ejecutó el AG en esta comparación.


## 16. Análisis y recomendación de métrica para el Rol 4

### Superficie de fitness

La superficie de fitness relaciona cada cromosoma con su
aptitud respecto a la referencia instrumental.

Los barridos del Rol 1 permiten observar ocho cortes
unidimensionales. No describen todas las interacciones
entre genes.

Una superficie relativamente suave facilita que pequeñas
variaciones produzcan cambios graduales de aptitud.
Un AG no necesita derivadas y la suavidad no garantiza
encontrar el óptimo global.

### Comparación de métricas

STFT compara magnitudes espectrales y su evolución temporal.
El croma compara la distribución de energía entre clases
de altura, con menor información directa sobre el timbre.

La comparación utiliza los mismos candidatos, melodía,
referencia y condiciones de síntesis.

Los rankings de sensibilidad describen qué genes afectan
más a cada métrica alrededor del cromosoma base.
Sus rangos numéricos no permiten declarar una métrica superior.

Las categorías de sensibilidad del módulo del Rol 1 utilizan
umbrales heurísticos; no son umbrales perceptuales validados.

### Recomendación inicial

Se propone mantener STFT como aptitud principal del AG
y croma como evaluación complementaria.

La recomendación debe considerar la escucha y la respuesta
a los genes. Una sensibilidad elevada al ruido de soplo
no demuestra por sí sola un mejor parecido instrumental.

### Limitaciones

- Se comparan seis candidatos, no una muestra exhaustiva.
- Los barridos modifican un gen cada vez.
- No se mide sensibilidad global ni interacción entre genes.
- La referencia depende del recorte, articulación y ganancia.
- Los silencios coincidentes influyen en las métricas.
- No se acredita superioridad perceptual sin una evaluación auditiva.
- Esta comparación no ejecuta el ciclo evolutivo del AG.

La siguiente celda genera una recomendación con los valores
calculados durante esta ejecución para compartir con el Rol 4.

In [18]:
from IPython.display import Markdown


fila_mejor_stft = comparacion.loc[
    comparacion["fitness_stft"].idxmax()
]

fila_mejor_croma = comparacion.loc[
    comparacion["fitness_croma"].idxmax()
]

gen_sensible_stft = ranking_sensibilidad.loc[
    ranking_sensibilidad["delta_fitness_stft"].idxmax()
]

gen_sensible_croma = ranking_sensibilidad.loc[
    ranking_sensibilidad["delta_fitness_croma"].idxmax()
]

coinciden = (
    fila_mejor_stft["candidato"]
    == fila_mejor_croma["candidato"]
)

conclusion_seleccion = (
    "Ambas métricas seleccionaron al mismo candidato."
    if coinciden
    else "Las métricas seleccionaron candidatos diferentes."
)

recomendacion = f"""# Rol 2 — Recomendación de métrica del Día 3

## Condiciones de comparación

- Candidatos evaluados: {len(comparacion)}.
- Semilla de candidatos: {SEMILLA_CANDIDATOS}.
- Semilla de síntesis: 2026.
- Genes analizados: {rep.num_genes()}.
- Puntos por gen: {PUNTOS_POR_GEN}.
- Evaluaciones de sensibilidad: {len(sensibilidad)}.
- Se utilizó la misma melodía y referencia para ambas métricas.

## Selección de candidatos

STFT seleccionó **{fila_mejor_stft['candidato']}**:

- Fitness STFT: {fila_mejor_stft['fitness_stft']:.6f}.
- Error espectral medio: {fila_mejor_stft['error_stft_db']:.6f} dB.

Croma seleccionó **{fila_mejor_croma['candidato']}**:

- Fitness croma: {fila_mejor_croma['fitness_croma']:.6f}.

{conclusion_seleccion}

Las puntuaciones STFT y croma no se comparan directamente
entre sí porque sus escalas y criterios son distintos.

## Respuesta a los genes

El mayor rango de variación STFT correspondió a
**{gen_sensible_stft['gen_nombre']}**:
{gen_sensible_stft['delta_fitness_stft']:.6f}.

El mayor rango de variación de croma correspondió a
**{gen_sensible_croma['gen_nombre']}**:
{gen_sensible_croma['delta_fitness_croma']:.6f}.

Son resultados de barridos individuales alrededor del
cromosoma por defecto. No describen sensibilidad global.

## Recomendación al Rol 4

Se propone usar `metrica="stft"` como aptitud principal
a maximizar y registrar croma como evaluación complementaria.

STFT permite evaluar diferencias espectrales relevantes
para el timbre. Croma aporta una comparación de alturas.

La recomendación es provisional hasta contrastarla con
la escucha y acordarla con el equipo. No se propone
un fitness combinado ni cambios de escala en esta etapa.

## Alcance

Esta comparación no ejecuta el AG y no demuestra convergencia
ni superioridad perceptual. El proyecto ya dispone de
`ejecutar_ag()` para realizar las corridas correspondientes.
"""

ruta_recomendacion = (
    CARPETA_AUDIO / "recomendacion_metricas_dia3.md"
)

ruta_recomendacion.write_text(
    recomendacion,
    encoding="utf-8",
)

display(Markdown(recomendacion))

print(
    "Recomendación guardada:",
    ruta_recomendacion.relative_to(RAIZ),
)

# Rol 2 — Recomendación de métrica del Día 3

## Condiciones de comparación

- Candidatos evaluados: 6.
- Semilla de candidatos: 42.
- Semilla de síntesis: 2026.
- Genes analizados: 8.
- Puntos por gen: 51.
- Evaluaciones de sensibilidad: 408.
- Se utilizó la misma melodía y referencia para ambas métricas.

## Selección de candidatos

STFT seleccionó **default**:

- Fitness STFT: 0.679606.
- Error espectral medio: 4.714410 dB.

Croma seleccionó **aleatorio_2**:

- Fitness croma: 0.920207.

Las métricas seleccionaron candidatos diferentes.

Las puntuaciones STFT y croma no se comparan directamente
entre sí porque sus escalas y criterios son distintos.

## Respuesta a los genes

El mayor rango de variación STFT correspondió a
**ruido_soplo**:
0.262472.

El mayor rango de variación de croma correspondió a
**relajacion_s**:
0.064137.

Son resultados de barridos individuales alrededor del
cromosoma por defecto. No describen sensibilidad global.

## Recomendación al Rol 4

Se propone usar `metrica="stft"` como aptitud principal
a maximizar y registrar croma como evaluación complementaria.

STFT permite evaluar diferencias espectrales relevantes
para el timbre. Croma aporta una comparación de alturas.

La recomendación es provisional hasta contrastarla con
la escucha y acordarla con el equipo. No se propone
un fitness combinado ni cambios de escala en esta etapa.

## Alcance

Esta comparación no ejecuta el AG y no demuestra convergencia
ni superioridad perceptual. El proyecto ya dispone de
`ejecutar_ag()` para realizar las corridas correspondientes.


Recomendación guardada: resultados\audios\recomendacion_metricas_dia3.md


## 17. Reproducibilidad y estado del Día 3

El Rol 2 reutiliza el esquema, los eventos y las funciones
de sensibilidad del Rol 1.

La comparación utiliza dos fuentes de evidencia:

- Seis candidatos evaluados con STFT y croma.
- Ocho barridos de 51 puntos, calculados mediante
  src/sensibilidad.py.

Los resultados propios de esta comparación se exportan
en resultados/audios/. Los archivos del Rol 1 en
resultados/sensibilidad/ se conservan.

El proyecto ya contiene ejecutar_ag() y los operadores
necesarios. Las celdas de comparación de este notebook
no ejecutan ese ciclo.

El registro del entorno describe versiones y parámetros.
No acredita por sí solo una ejecución desde un kernel limpio.

Antes de cerrar el Día 3:

- Reiniciar el kernel y ejecutar todas las celdas.
- Revisar los audios seleccionados por ambas métricas.
- Confirmar tablas, figura y recomendación exportadas.
- Compartir la recomendación con el Rol 4.
- Registrar si el equipo acepta STFT como criterio principal.
- Guardar las salidas actualizadas del notebook.

La comprobación de una página, nombres, firmas y fecha
corresponde al documento de diseño del equipo.

In [19]:
import platform
import subprocess
import hashlib

from importlib.metadata import (
    PackageNotFoundError,
    version,
)

import src.fitness as modulo_fitness


def obtener_version(paquete):
    try:
        return version(paquete)
    except PackageNotFoundError:
        return "No disponible"


versiones = {
    "Python": platform.python_version(),
    **{
        paquete: obtener_version(paquete)
        for paquete in [
            "numpy",
            "pandas",
            "scipy",
            "soundfile",
            "matplotlib",
            "ipython",
            "ipykernel",
        ]
    },
}


# El ZIP puede no contener metadatos de Git.
commit = None
cambios_locales = None

try:
    raiz_git = subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        cwd=RAIZ,
        capture_output=True,
        text=True,
        check=True,
    )

    if Path(raiz_git.stdout.strip()).resolve() == RAIZ:
        resultado_commit = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=RAIZ,
            capture_output=True,
            text=True,
            check=True,
        )

        resultado_estado = subprocess.run(
            ["git", "status", "--porcelain"],
            cwd=RAIZ,
            capture_output=True,
            text=True,
            check=True,
        )

        commit = resultado_commit.stdout.strip()
        cambios_locales = bool(
            resultado_estado.stdout.strip()
        )

except (OSError, subprocess.CalledProcessError):
    pass


registro_ejecucion = {
    "versiones": versiones,
    "commit": commit,
    "cambios_locales": cambios_locales,
    "semilla_candidatos": SEMILLA_CANDIDATOS,
    "semilla_sintesis": 2026,
    "frecuencia_muestreo": int(FRECUENCIA_MUESTREO),
    "n_fft": int(modulo_fitness.N_FFT),
    "salto": int(modulo_fitness.SALTO),
    "piso_db": float(modulo_fitness.PISO_DB),
    "escala_error_db": float(
        modulo_fitness.ESCALA_ERROR_DB
    ),
    "genes": rep.obtener_nombres_genes(),
    "gen_default": np.asarray(
        gen_prueba,
        dtype=float,
    ).tolist(),
    "notas": notas.astype(int).tolist(),
    "duraciones_s": duraciones.tolist(),
    "cantidad_candidatos": len(candidatos),
    "gen_barrido": GEN_BARRIDO,
    "cantidad_puntos_barrido": len(barrido),
    "mejor_candidato_stft": str(nombre_mejor),
    "referencia": str(
        ruta_referencia.relative_to(RAIZ)
    ),
    "carpeta_resultados": str(
        CARPETA_AUDIO.relative_to(RAIZ)
    ),
}

registro_ejecucion.update({
    "etapa": "Rol 2 - Día 3",
    "modulo_sensibilidad": "src.sensibilidad",
    "puntos_por_gen": PUNTOS_POR_GEN,
    "evaluaciones_sensibilidad": len(sensibilidad),
    "mejor_candidato_croma": nombre_mejor_croma,
    "recomendacion_metricas": str(
        ruta_recomendacion.relative_to(RAIZ)
    ),
    "melodia_sha256": hashlib.sha256(
        ruta_csv.read_bytes()
    ).hexdigest(),
    "referencia_sha256": hashlib.sha256(
        ruta_referencia.read_bytes()
    ).hexdigest(),
})

ruta_registro = (
    CARPETA_AUDIO / "registro_ejecucion_rol2.json"
)

ruta_registro.write_text(
    json.dumps(
        registro_ejecucion,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

display(pd.DataFrame(
    versiones.items(),
    columns=["componente", "version"],
))

print(
    "Registro guardado:",
    ruta_registro.relative_to(RAIZ),
)

print(
    "Commit:",
    commit if commit is not None else "No disponible",
)

print(
    "Cambios locales:",
    cambios_locales
    if cambios_locales is not None
    else "No comprobado",
)

print(
    "Comparación del Día 3 completada. "
    "Estas celdas no ejecutaron el ciclo del AG."
)

,componente,version
0,Python,3.13.2
1,numpy,2.2.4
2,pandas,2.2.3
3,scipy,1.16.1
4,soundfile,0.14.0
5,matplotlib,3.10.1
6,ipython,9.0.2
7,ipykernel,6.29.5


Registro guardado: resultados\audios\registro_ejecucion_rol2.json
Commit: ebb344d87fb5500feeaef8b01a6a6de5f7fe9dbb
Cambios locales: True
Comparación del Día 3 completada. Estas celdas no ejecutaron el ciclo del AG.


## 18. Día 4 — Superficie de fitness y documentación de la evaluación

### Síntesis y evaluación utilizadas

El Rol 2 reutiliza los ocho genes normalizados del Rol 1, su
validación y su decodificación a parámetros físicos.

La síntesis combina tres armónicos, una envolvente ADSR y ruido
de soplo. Conserva las notas, duraciones y silencios del CSV.
La semilla de síntesis es 2026 y permanece independiente del
generador aleatorio del AG.

La evaluación STFT utiliza ventana Hann, FFT de 2048 muestras,
salto de 256 muestras, referencia fija de amplitud 1 y piso
de −80 dB. Su error absoluto medio se transforma en aptitud:

fitness_stft = 1 / (1 + error_stft_db / 10)

La aptitud se maximiza. El factor 10 establece una escala,
sin representar un umbral perceptual validado.

El croma compara la distribución de energía en doce clases
de altura mediante similitud coseno por cuadro temporal.
Se conserva como evaluación complementaria.

### Pregunta teórica 2

¿Qué es una superficie de fitness? ¿Por qué es importante
que sea suave para un AG? Ilustra con un gráfico del proyecto.

Una superficie de fitness relaciona cada cromosoma con la
aptitud que obtiene frente a una referencia. En este proyecto,
las ocho coordenadas del cromosoma controlan parámetros del
instrumento y la aptitud expresa su parecido según la métrica.

Una superficie relativamente suave permite que pequeños
cambios en los genes produzcan cambios graduales de aptitud.
Esto facilita los ajustes locales mediante mutación y ayuda
a conservar mejoras durante la búsqueda.

El AG no necesita derivadas ni exige suavidad matemática.
Una superficie suave también puede contener varios óptimos
locales y no garantiza encontrar el óptimo global.

### Interpretación de la figura

Se reutilizan los barridos calculados en la sección 14 mediante
src/sensibilidad.py del Rol 1.

La figura presenta dos cortes de la superficie STFT:

- ruido_soplo.
- amp_arm2.

Cada corte modifica un solo gen en 51 valores normalizados,
manteniendo fijos los otros siete genes, la melodía,
la referencia y la semilla de síntesis.

Las escalas verticales son diferentes para observar la forma
de cada curva. Deben considerarse sus valores al comparar
la magnitud de las variaciones.

Los cortes no prueban suavidad global, ausencia de óptimos
locales ni independencia entre genes. Las líneas unen puntos
evaluados y no demuestran el comportamiento entre esos puntos.

La codificación lineal del Rol 1 conserva proximidad entre
parámetros físicos, pero no garantiza suavidad de toda la
cadena de síntesis y evaluación. La envolvente por tramos,
su ajuste en notas cortas y el piso de dB pueden introducir
cambios de pendiente.

### Recomendación y alcance

Se propone mantener STFT como aptitud principal a maximizar
y croma como evaluación complementaria. Esta recomendación
sigue pendiente de escucha y ratificación colectiva.

La comparación de candidatos y los barridos no ejecutan
el ciclo completo del AG.

La siguiente celda exporta la figura y un resumen calculado
con los resultados disponibles en esta ejecución. El tipo
de ejecutor registrado no certifica por sí solo que se haya
reiniciado el kernel y ejecutado todo en orden.

In [20]:
import hashlib
import json

import matplotlib.pyplot as plt

from IPython import get_ipython
from IPython.display import Markdown, display


# Reutilizar los resultados calculados en las secciones anteriores.
GENES_FIGURA_DIA4 = (
    "ruido_soplo",
    "amp_arm2",
)

figura_dia4, ejes_dia4 = plt.subplots(
    1,
    2,
    figsize=(11, 4),
)

resumen_cortes_dia4 = []

for eje, nombre_gen in zip(
    ejes_dia4,
    GENES_FIGURA_DIA4,
):
    corte = (
        sensibilidad.loc[
            sensibilidad["gen_nombre"] == nombre_gen
        ]
        .sort_values("valor_normalizado")
        .copy()
    )

    assert len(corte) == PUNTOS_POR_GEN

    assert np.all(np.isfinite(
        corte[
            ["valor_normalizado", "fitness_stft"]
        ].to_numpy()
    ))

    eje.plot(
        corte["valor_normalizado"],
        corte["fitness_stft"],
        color="#175C88",
        linewidth=2,
    )

    eje.set(
        title=nombre_gen,
        xlabel="Valor normalizado [0, 1]",
        ylabel="Fitness STFT (mayor es mejor)",
    )

    eje.grid(alpha=0.25)

    resumen_cortes_dia4.append({
        "gen": nombre_gen,
        "puntos": int(len(corte)),
        "fitness_stft_min": float(
            corte["fitness_stft"].min()
        ),
        "fitness_stft_max": float(
            corte["fitness_stft"].max()
        ),
        "rango_fitness_stft": float(
            np.ptp(corte["fitness_stft"].to_numpy())
        ),
    })


figura_dia4.suptitle(
    "Cortes 1D de la superficie de fitness: "
    "escalas Y diferentes"
)

figura_dia4.tight_layout()

ruta_figura_dia4 = (
    CARPETA_AUDIO / "rol2_dia4_superficie_fitness.png"
)

figura_dia4.savefig(
    ruta_figura_dia4,
    dpi=180,
    bbox_inches="tight",
)

display(figura_dia4)
plt.close(figura_dia4)


# Obtener los ganadores desde la comparación de esta ejecución.
fila_stft_dia4 = comparacion.loc[
    comparacion["fitness_stft"].idxmax()
]

fila_croma_dia4 = comparacion.loc[
    comparacion["fitness_croma"].idxmax()
]

entorno_interactivo = get_ipython()

tipo_ejecucion_dia4 = (
    "kernel_jupyter"
    if getattr(entorno_interactivo, "kernel", None)
    else "proceso_python"
)

resumen_dia4 = {
    "etapa": "Rol 2 - Día 4",
    "origen": (
        "Resultados disponibles al ejecutar "
        "las celdas del notebook"
    ),
    "tipo_ejecucion": tipo_ejecucion_dia4,
    "kernel_limpio_certificado": False,
    "cantidad_genes": int(rep.num_genes()),
    "cantidad_candidatos": int(len(comparacion)),
    "puntos_por_gen": int(PUNTOS_POR_GEN),
    "evaluaciones_sensibilidad": int(len(sensibilidad)),
    "semilla_candidatos": int(SEMILLA_CANDIDATOS),
    "semilla_sintesis": 2026,
    "ganador_stft": str(
        fila_stft_dia4["candidato"]
    ),
    "fitness_stft_ganador": float(
        fila_stft_dia4["fitness_stft"]
    ),
    "error_stft_db_ganador": float(
        fila_stft_dia4["error_stft_db"]
    ),
    "ganador_croma": str(
        fila_croma_dia4["candidato"]
    ),
    "fitness_croma_ganador": float(
        fila_croma_dia4["fitness_croma"]
    ),
    "cortes": resumen_cortes_dia4,
    "figura": (
        ruta_figura_dia4
        .relative_to(RAIZ)
        .as_posix()
    ),
    "melodia_sha256": hashlib.sha256(
        ruta_csv.read_bytes()
    ).hexdigest(),
    "referencia_sha256": hashlib.sha256(
        ruta_referencia.read_bytes()
    ).hexdigest(),
    "ag_ejecutado_en_esta_comparacion": False,
    "recomendacion": (
        "STFT principal y croma complementario"
    ),
    "ratificacion_equipo": "pendiente",
}

ruta_resumen_dia4 = (
    CARPETA_AUDIO / "rol2_dia4_resumen.json"
)

ruta_resumen_dia4.write_text(
    json.dumps(
        resumen_dia4,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    ),
    encoding="utf-8",
)

display(pd.DataFrame(resumen_cortes_dia4))

display(Markdown(
    f"**Resultados del Día 4**\n\n"
    f"- Candidatos comparados: {len(comparacion)}.\n"
    f"- Evaluaciones de sensibilidad: "
    f"{len(sensibilidad)}.\n"
    f"- Seleccionado por STFT: "
    f"**{fila_stft_dia4['candidato']}**.\n"
    f"- Seleccionado por croma: "
    f"**{fila_croma_dia4['candidato']}**.\n\n"
    "Los cortes describen variaciones individuales de genes. "
    "La escucha y la aprobación colectiva siguen pendientes. "
    "Esta comparación no ejecuta el ciclo del AG."
))

print(
    "Figura guardada:",
    ruta_figura_dia4.relative_to(RAIZ),
)

print(
    "Resumen guardado:",
    ruta_resumen_dia4.relative_to(RAIZ),
)

print(
    "Tipo de ejecución detectado:",
    tipo_ejecucion_dia4,
)

<Figure size 1100x400 with 2 Axes>

,gen,puntos,fitness_stft_min,fitness_stft_max,rango_fitness_stft
0,ruido_soplo,51,0.448903,0.711375,0.262472
1,amp_arm2,51,0.677237,0.683801,0.006564


**Resultados del Día 4**

- Candidatos comparados: 6.
- Evaluaciones de sensibilidad: 408.
- Seleccionado por STFT: **default**.
- Seleccionado por croma: **aleatorio_2**.

Los cortes describen variaciones individuales de genes. La escucha y la aprobación colectiva siguen pendientes. Esta comparación no ejecuta el ciclo del AG.

Figura guardada: resultados\audios\rol2_dia4_superficie_fitness.png
Resumen guardado: resultados\audios\rol2_dia4_resumen.json
Tipo de ejecución detectado: kernel_jupyter
